In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하는 것을 권장합니다.

```
qiskit[all]~=2.3.0
```
</details>
$n$개의 비트(또는 Qubit)가 있을 때, 각 비트에는 보통 $0 \rightarrow n-1$의 레이블을 붙입니다. 각 소프트웨어와 리소스는 컴퓨터 메모리 및 화면 표시 시 이 비트들을 어떤 순서로 배열할지 결정해야 합니다.

## Qiskit 규칙

다음은 Qiskit SDK가 다양한 시나리오에서 비트를 정렬하는 방법입니다.

### Quantum Circuit

`QuantumCircuit` 클래스는 Qubit를 리스트(`QuantumCircuit.qubits`)에 저장합니다. 이 리스트에서 Qubit의 인덱스가 해당 Qubit의 레이블을 정의합니다.

In [1]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit import Qubit

qc = QuantumCircuit(2)
qc.qubits[0]  # qubit "0"

Qubit(QuantumRegister(2, "q"), 0)

<Qubit register=(2, "q"), index=0>

### Circuit diagrams

On a circuit diagram, qubit $0$ is the topmost qubit, and qubit $n-1$ the
bottommost qubit. You can change this with the `reverse_bits` argument of
`QuantumCircuit.draw` (see [Change ordering in
Qiskit](#change-ordering-in-qiskit)).

In [2]:
qc.x(1)
qc.draw()

          
q_0: ─────
     ┌───┐
q_1: ┤ X ├
     └───┘

### Circuit 다이어그램
Circuit 다이어그램에서 Qubit $0$은 가장 위쪽 Qubit이고, Qubit $n-1$은 가장 아래쪽 Qubit입니다. `QuantumCircuit.draw`의 `reverse_bits` 인수를 사용하여 이 순서를 변경할 수 있습니다([Qiskit에서 순서 변경하기](#change-ordering-in-qiskit) 참조).

In [3]:
from qiskit.primitives import StatevectorSampler as Sampler

qc.measure_all()

job = Sampler().run([qc])
result = job.result()
print(f" > Counts: {result[0].data.meas.get_counts()}")

 > Counts: {'10': 1024}


### Strings

When displaying or interpreting a list of bits (or qubits) as a string, bit
$n-1$ is the leftmost bit, and bit $0$ is the rightmost bit. This is because we
usually write numbers with the most significant digit on the left, and in
Qiskit, bit $n-1$ is interpreted as the most significant bit.

For example, the following cell defines a `Statevector` from a string of
single-qubit states. In this case, qubit $0$ is in state $|+\rangle$, and
qubit $1$ in state $|0\rangle$.

In [4]:
from qiskit.quantum_info import Statevector

sv = Statevector.from_label("0+")
sv.probabilities_dict()

{np.str_('00'): np.float64(0.4999999999999999),
 np.str_('01'): np.float64(0.4999999999999999)}

### 정수
비트를 숫자로 해석할 때, 비트 $0$은 최하위 비트(least significant bit)이고, 비트 $n-1$은 최상위 비트(most significant bit)입니다. 이는 코딩 시 각 비트가 $2^\text{label}$ 값을 갖기 때문에 편리합니다(레이블은 `QuantumCircuit.qubits`에서 Qubit의 인덱스입니다). 예를 들어, 다음 Circuit 실행은 비트 $0$이 `0`이고 비트 $1$이 `1`인 상태로 종료됩니다. 이는 십진수 정수 `2`로 해석됩니다(측정 확률 `1.0`).

In [5]:
print(sv[1])  # amplitude of state |01>
print(sv[2])  # amplitude of state |10>

(0.7071067811865475+0j)
0j


### Gates

Each gate in Qiskit can interpret a list of qubits in its own way, but
controlled gates usually follow the convention `(control, target)`.

For example, the following cell adds a controlled-X gate where qubit $0$ is
the control and qubit $1$ is the target.

In [6]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.cx(0, 1)
qc.draw()

          
q_0: ──■──
     ┌─┴─┐
q_1: ┤ X ├
     └───┘

### 문자열
비트(또는 Qubit) 리스트를 문자열로 표시하거나 해석할 때, 비트 $n-1$은 가장 왼쪽 비트이고 비트 $0$은 가장 오른쪽 비트입니다. 이는 숫자를 쓸 때 보통 최상위 숫자를 왼쪽에 쓰기 때문이며, Qiskit에서 비트 $n-1$은 최상위 비트로 해석됩니다.

예를 들어, 다음 셀은 단일 Qubit 상태의 문자열로부터 `Statevector`를 정의합니다. 이 경우, Qubit $0$은 $|+\rangle$ 상태이고 Qubit $1$은 $|0\rangle$ 상태입니다.

In [7]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.x(0)
qc.draw(reverse_bits=True)

          
q_1: ─────
     ┌───┐
q_0: ┤ X ├
     └───┘

You can use the `reverse_bits` method to return a new circuit with the
qubits' labels reversed (this does not mutate the original circuit).

In [8]:
qc.reverse_bits().draw()

          
q_0: ─────
     ┌───┐
q_1: ┤ X ├
     └───┘

가장 왼쪽 비트가 비트 $0$일 것으로 예상할 수 있지만 실제로는 비트 $n-1$을 나타내기 때문에, 비트 문자열을 해석할 때 간혹 혼란이 생길 수 있습니다.

### Statevector 행렬
상태 벡터를 복소수(진폭) 리스트로 나타낼 때, Qiskit은 인덱스 $x$의 진폭이 계산 기저 상태 $|x\rangle$를 나타내도록 진폭을 정렬합니다.